In [ ]:
!pip install matplotlib

In [ ]:
!pip install scipy
!pip install pyarrow

In [ ]:
!pip install --upgrade pyarrow

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_ROOT = PROJECT_ROOT / "data" / "raw"

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PHASE2_DIR = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE2_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT, "| exists:", RAW_ROOT.exists())
print("PHASE2_DIR:", PHASE2_DIR)


In [ ]:
QC_ALL_PATH = PROJECT_ROOT / "data" / "processed" / "qc" / "00_QC_ALL_SUBJECTS.csv"
print("QC_ALL_PATH:", QC_ALL_PATH, "| exists:", QC_ALL_PATH.exists())

def load_qc_subject(qc_all_path: Path, subject_name: str) -> pd.DataFrame:
    qc_all = pd.read_csv(qc_all_path)
    qc_subj = qc_all[qc_all["subject"].astype(str) == str(subject_name)].copy()
    if qc_subj.empty:
        raise FileNotFoundError(f"No QC rows found for subject '{subject_name}' in {qc_all_path.name}")
    return qc_subj

def compute_trial_status(qc_subj: pd.DataFrame, gap_thr=0, sat_thr=0.01) -> pd.DataFrame:
    qc = qc_subj.copy()
    qc["qc_fail"] = (
        (qc["mapping_ok"].fillna(True) == False) |
        (qc["emg_gaps"].fillna(0) > gap_thr) |
        (qc["imu_gyro_mag_gaps"].fillna(0) > gap_thr) |
        (qc["imu_acc_mag_gaps"].fillna(0) > gap_thr) |
        (
            (qc["emg_n_samples"].fillna(0) > 0) & (
                (qc["emg_lowvar_flag"].fillna(False) == True) |
                (qc["emg_saturation_ratio"].fillna(0) > sat_thr)
            )
        )
    )
    trial_status = qc.groupby("trial_id")["qc_fail"].any().reset_index()
    trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")
    return trial_status


In [ ]:
import sys, inspect, importlib

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import trigno_io
importlib.reload(trigno_io)
from trigno_io import read_trigno_csv

print("read_trigno_csv imported from:", inspect.getfile(read_trigno_csv))


In [ ]:
from scipy.signal import butter, filtfilt, iirnotch, welch

def butter_bandpass(low_hz, high_hz, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low_hz/nyq, high_hz/nyq], btype="bandpass")
    return b, a

def butter_lowpass(cutoff_hz, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff_hz/nyq, btype="lowpass")
    return b, a

def notch_50hz(fs, q=30.0, f0=50.0):
    w0 = f0 / (fs/2.0)
    b, a = iirnotch(w0=w0, Q=q)
    return b, a

def has_50hz_peak(x, fs, band=(45, 55), prominence_ratio=5.0):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < int(fs * 2):
        return False

    f, Pxx = welch(x, fs=fs, nperseg=min(len(x), int(fs*2)))
    band_mask = (f >= band[0]) & (f <= band[1])
    ref_mask  = (f >= 60) & (f <= 80)

    if not np.any(band_mask) or not np.any(ref_mask):
        return False

    band_pow = np.mean(Pxx[band_mask])
    ref_pow  = np.mean(Pxx[ref_mask]) + 1e-12
    return (band_pow / ref_pow) >= prominence_ratio

def preprocess_emg_signal(x, fs_emg,
                          bp_low=20, bp_high=450, bp_order=4,
                          apply_notch_if_peak=True, notch_q=30.0,
                          env_lp=6.0, env_order=4):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    b_bp, a_bp = butter_bandpass(bp_low, bp_high, fs_emg, order=bp_order)
    emg_bp = filtfilt(b_bp, a_bp, x)

    notch_applied = False
    if apply_notch_if_peak:
        if has_50hz_peak(emg_bp, fs_emg):
            b_n, a_n = notch_50hz(fs_emg, q=notch_q, f0=50.0)
            emg_bp = filtfilt(b_n, a_n, emg_bp)
            notch_applied = True

    rect = np.abs(emg_bp)
    b_lp, a_lp = butter_lowpass(env_lp, fs_emg, order=env_order)
    env = filtfilt(b_lp, a_lp, rect)

    return emg_bp, env, notch_applied


In [ ]:
def saturation_mask(x, run_len_samples=500, tol=0.0):
    x = np.asarray(x, dtype=float)
    mask = np.zeros(len(x), dtype=bool)

    finite = np.isfinite(x)
    if np.sum(finite) < run_len_samples + 1:
        return mask

    x2 = x.copy()
    x2[~finite] = np.nan

    dx = np.diff(x2)
    if tol == 0.0:
        same = (dx == 0)
    else:
        same = (np.abs(dx) <= tol)

    start = None
    count = 0
    for i, s in enumerate(same):
        if s and np.isfinite(dx[i]):
            if start is None:
                start = i
                count = 2
            else:
                count += 1
        else:
            if start is not None and count >= run_len_samples:
                mask[start:start+count] = True
            start = None
            count = 0

    if start is not None and count >= run_len_samples:
        mask[start:start+count] = True

    return mask

def expand_mask_in_time(mask, fs, pad_seconds=0.5):
    if pad_seconds <= 0:
        return mask
    pad = int(round(pad_seconds * fs))
    if pad <= 0:
        return mask
    out = mask.copy()
    idx = np.where(mask)[0]
    for i in idx:
        lo = max(0, i - pad)
        hi = min(len(out), i + pad + 1)
        out[lo:hi] = True
    return out


In [ ]:
REPAIR_TRIALS = {
    # example:
    "Healthy_Subject_5": {"lifting_exo"},
    # if needed later:
    # "ALS_Subject_10": {"drinking_noexo", "lifting_noexo"},
}



In [ ]:
def subject_csv_files(subject_name: str) -> list[Path]:
    subject_dir = RAW_ROOT / subject_name / "EMG&IMUTest"
    if not subject_dir.exists():
        raise FileNotFoundError(f"Subject dir not found: {subject_dir}")
    return sorted(subject_dir.glob("*.csv"))


In [ ]:
def preprocess_trial_to_long_df(
    subject_name: str,
    csv_path: Path,
    qc_subj: pd.DataFrame,
    bp_low=20, bp_high=450, env_lp=6.0,
    notch_conditional=True,
    fs_fallback=1259.259259,
    repair_pad_seconds=0.5,
    sat_run_len=500,
    sat_tol=0.0
):
    sensors, per_sensor, meta = read_trigno_csv(csv_path)
    trial_id = csv_path.stem

    rows = []
    notch_log = []
    skip_log = []

    for sensor in sensors:
        emg = per_sensor[sensor].get("emg", None)

        # skip sensors with no EMG (imu_only etc.)
        if emg is None or len(emg) == 0:
            skip_log.append({
                "subject": subject_name,
                "trial_id": trial_id,
                "sensor": sensor,
                "reason": "no_emg_in_reader_layout",
                "layout": meta.get("layout", ""),
                "block_kinds": meta.get("block_kinds", "")
            })
            continue

        t = emg["t_emg"].values
        x = emg["emg_mv"].values

        # fs from QC if available
        qrow = qc_subj[(qc_subj["trial_id"] == trial_id) & (qc_subj["sensor"] == sensor)]
        if len(qrow) == 1 and np.isfinite(qrow["emg_fs_est"].values[0]):
            fs_emg = float(qrow["emg_fs_est"].values[0])
        else:
            fs_emg = fs_fallback

        # preprocess
        emg_bp, env, notch_applied = preprocess_emg_signal(
            x, fs_emg,
            bp_low=bp_low, bp_high=bp_high,
            apply_notch_if_peak=notch_conditional,
            env_lp=env_lp
        )

        # partial repair: mask saturated segments ONLY for selected trials
        if trial_id in REPAIR_TRIALS.get(subject_name, set()):
            m = saturation_mask(x, run_len_samples=sat_run_len, tol=sat_tol)
            m = expand_mask_in_time(m, fs=fs_emg, pad_seconds=repair_pad_seconds)
            if np.any(m):
                emg_bp = emg_bp.copy()
                env = env.copy()
                emg_bp[m] = np.nan
                env[m] = np.nan

        notch_log.append({
            "subject": subject_name,
            "trial_id": trial_id,
            "sensor": sensor,
            "fs_emg_used": fs_emg,
            "notch_applied": notch_applied
        })

        df_sensor = pd.DataFrame({
            "subject": subject_name,
            "trial_id": trial_id,
            "sensor": sensor,
            "t_emg": t,
            "emg_bp": emg_bp,
            "env": env
        })
        rows.append(df_sensor)

    out = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    notch_df = pd.DataFrame(notch_log)
    skip_df = pd.DataFrame(skip_log)
    return out, notch_df, skip_df


In [ ]:
def run_phase2_for_subject(
    subject_name: str,
    bp_low: float = 20,
    bp_high: float = 450,
    env_lp: float = 6.0,
    notch_conditional: bool = True,
    gap_thr: int = 0,
    sat_thr: float = 0.01,
    fs_fallback: float = 1259.259259,
    repair_pad_seconds: float = 0.5,
):
    files = subject_csv_files(subject_name)
    print("Files found:", len(files))

    qc_subj = load_qc_subject(QC_ALL_PATH, subject_name)
    trial_status = compute_trial_status(qc_subj, gap_thr=gap_thr, sat_thr=sat_thr)

    ok_trials = set(trial_status.loc[trial_status["status"] == "OK", "trial_id"].astype(str))
    repair_trials = REPAIR_TRIALS.get(subject_name, set())

    allowed_trials = ok_trials | set(repair_trials)
    print(f"OK trials: {len(ok_trials)} | FAIL trials: {int((trial_status['status']=='FAIL').sum())}")
    if repair_trials:
        print(f"Repair-enabled trials for {subject_name}: {sorted(repair_trials)}")

    files_allowed = [fp for fp in files if fp.stem in allowed_trials]
    print("Files to preprocess (OK + repair):", len(files_allowed))

    all_long = []
    notch_logs = []
    skip_logs = []

    for fp in files_allowed:
        try:
            long_df, notch_df, skip_df = preprocess_trial_to_long_df(
                subject_name=subject_name,
                csv_path=fp,
                qc_subj=qc_subj,
                bp_low=bp_low,
                bp_high=bp_high,
                env_lp=env_lp,
                notch_conditional=notch_conditional,
                fs_fallback=fs_fallback,
                repair_pad_seconds=repair_pad_seconds
            )
        except Exception as e:
            print(f"[SKIP FILE] {fp.name} due to {type(e).__name__}: {e}")
            continue

        if not long_df.empty:
            all_long.append(long_df)
        if not notch_df.empty:
            notch_logs.append(notch_df)
        if not skip_df.empty:
            skip_logs.append(skip_df)

    subj_long = pd.concat(all_long, ignore_index=True) if all_long else pd.DataFrame()
    notch_all = pd.concat(notch_logs, ignore_index=True) if notch_logs else pd.DataFrame()
    skip_all  = pd.concat(skip_logs, ignore_index=True) if skip_logs else pd.DataFrame()

    # summaries for display (no saving)
    if not notch_all.empty:
        notch_summary = (
            notch_all.groupby("trial_id")["notch_applied"]
            .mean().reset_index()
            .rename(columns={"notch_applied": "notch_applied_ratio"})
            .sort_values("trial_id")
        )
    else:
        notch_summary = pd.DataFrame(columns=["trial_id", "notch_applied_ratio"])

    if not skip_all.empty:
        skip_summary = (
            skip_all.groupby("reason")["sensor"]
            .count().reset_index()
            .rename(columns={"sensor": "n_skipped"})
            .sort_values("n_skipped", ascending=False)
        )
    else:
        skip_summary = pd.DataFrame(columns=["reason", "n_skipped"])

    # save ONLY main parquet
    out_parquet = PHASE2_DIR / f"{subject_name}__emg_bp_env.parquet"
    if subj_long.empty:
        print("WARNING: No EMG rows produced. Nothing saved.")
    else:
        subj_long.to_parquet(out_parquet, index=False)
        print("Saved:", out_parquet)

    return subj_long, trial_status, notch_summary, skip_summary


In [ ]:
SUBJECT_NAME = "ALS_Subject_1"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


subject 2

In [ ]:
SUBJECT_NAME = "ALS_Subject_2"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())



In [ ]:
SUBJECT_NAME = "ALS_Subject_3"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_4"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_5"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_7"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_8"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_9"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_10"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_11"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_12"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_13"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_14"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_15"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "ALS_Subject_16"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "Healthy_Subject_1"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "Healthy_Subject_2"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "Healthy_Subject_3"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "Healthy_Subject_4"

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "Healthy_Subject_5"   # change only this each run

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01,
    repair_pad_seconds=0.5
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "Healthy_Subject_6"   # change only this each run

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01,
    repair_pad_seconds=0.5
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "Healthy_Subject_7"   # change only this each run

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01,
    repair_pad_seconds=0.5
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())


In [ ]:
SUBJECT_NAME = "Healthy_Subject_8"   # change only this each run

subj_long, trial_status, notch_summary, skip_summary = run_phase2_for_subject(
    SUBJECT_NAME,
    bp_low=20, bp_high=450,
    env_lp=6.0,
    notch_conditional=True,
    gap_thr=0,
    sat_thr=0.01,
    repair_pad_seconds=0.5
)

display(trial_status.sort_values("trial_id"))
display(notch_summary)
display(skip_summary)
display(subj_long.head())